In [1]:
# 03_make_weights.py
from pathlib import Path
import pandas as pd
import numpy as np

# === cấu hình ===
port_tickers = ['AAPL','MSFT','NVDA','GOOGL','AMZN','TSLA','JPM','JNJ','UNH','XOM','PG','CAT','NEE']
proc_calendar_path = Path('data/processed/calendar.parquet')

out_weights_long = Path('data/portfolio_weights/ew_monthly.parquet')
out_weights_wide = Path('data/portfolio_weights/ew_monthly_wide.parquet')
out_weights_long.parent.mkdir(parents=True, exist_ok=True)

# === đọc calendar ===
calendar = pd.read_parquet(proc_calendar_path)
calendar['date'] = pd.to_datetime(calendar['date'])
calendar = calendar.set_index('date')

# === xác định ngày tái cân bằng: ngày giao dịch đầu tiên mỗi tháng ===
month_first = calendar.index.to_series().groupby([calendar.index.year, calendar.index.month]).min()
rebal_days = month_first.sort_values().tolist()

# === tạo bảng trọng số ở ngày tái cân bằng (mỗi ticker = 1/N) ===
N = len(port_tickers)
w_rebal = pd.DataFrame(index=rebal_days, columns=port_tickers, data=1.0/N)

# === mở rộng ra toàn bộ ngày giao dịch bằng ffill (piecewise-constant) ===
w_wide = w_rebal.reindex(calendar.index).ffill()
# chuẩn hóa đề phòng sai số số học
w_wide = w_wide.div(w_wide.sum(axis=1), axis=0)

# === lưu WIDE (mỗi cột là 1 ticker) ===
w_wide.to_parquet(out_weights_wide)
print(f'✓ saved weights wide -> {out_weights_wide.resolve()}')

# === tạo LONG (date, ticker, weight) để thuận tiện groupby/pivot sau này ===
w_long = w_wide.reset_index().melt(id_vars='date', var_name='ticker', value_name='weight')
w_long.to_parquet(out_weights_long, index=False)
print(f'✓ saved weights long -> {out_weights_long.resolve()}')

# === kiểm tra logic: tổng trọng số mỗi ngày = 1 ===
s = w_wide.sum(axis=1)
max_err = float(np.abs(s - 1.0).max())
print('max |sum(weights)-1| =', max_err)

✓ saved weights wide -> C:\Users\ACER\python\Dissertation\Port Dashboard\data\portfolio_weights\ew_monthly_wide.parquet
✓ saved weights long -> C:\Users\ACER\python\Dissertation\Port Dashboard\data\portfolio_weights\ew_monthly.parquet
max |sum(weights)-1| = 4.440892098500626e-16


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

w_wide = pd.read_parquet("data/portfolio_weights/ew_monthly_wide.parquet")
w_long = pd.read_parquet("data/portfolio_weights/ew_monthly.parquet")

# 13 tickers và không có SPY
assert 'SPY' not in w_wide.columns, "Weights không được chứa SPY"
assert w_wide.shape[1] == 13, "Số cột weights phải = 13 tickers"
assert not w_wide.isna().any().any(), "Weights có NaN"

# sum = 1
s = w_wide.sum(axis=1)
max_err = float(np.abs(s - 1.0).max())
print("max |sum-1| =", max_err)
assert max_err < 1e-9, "Tổng trọng số != 1"

# chỉ đổi ở ngày đầu tháng
idx = w_wide.index
month_key = idx.to_period('M')
change_days = (w_wide.ne(w_wide.shift(1))).any(axis=1)
changed = w_wide[change_days]
# mọi change day phải là ngày đầu tháng
is_month_start = changed.index.isin(
    idx.to_series().groupby([idx.year, idx.month]).min().tolist()
)
assert is_month_start.all(), "Weights thay đổi không đúng ngày đầu tháng"

# long format khớp wide
re_melt = w_wide.reset_index().melt(id_vars='date', var_name='ticker', value_name='weight')
re_melt['date'] = pd.to_datetime(re_melt['date'])
w_long['date'] = pd.to_datetime(w_long['date'])
re_melt = re_melt.sort_values(['date','ticker']).reset_index(drop=True)
w_long  = w_long.sort_values(['date','ticker']).reset_index(drop=True)
pd.testing.assert_series_equal(re_melt['weight'].round(12), w_long['weight'].round(12))

print("WEIGHTS: OK ✅")

max |sum-1| = 4.440892098500626e-16
WEIGHTS: OK ✅
